# 2.1 • From georeferenced pixels to model tensors

**Learning objectives:** inspect a raster transform; extract a window; trace channel/batch dimensions; distinguish pretrained features from class predictions.

**How to learn:** predict → run → explain → change one thing. Spend 45–90 minutes here.

**Data:** the default worked example is deterministic synthetic practice data near Colorado. It is not SNOTEL, a watershed survey, a burn map, or a foundation-model output.


## Before you run

A raster has array indices AND a mapping to coordinates. A model sees a tensor shaped batch × channels × height × width. Our synthetic raster teaches those mechanics without a download. The optional TorchGeo lab requires `requirements-models.txt`.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import OUT, points


## Create and inspect a tiny georeferenced raster


In [ ]:
import rasterio
from rasterio.windows import Window
from course import raster_fixture
path=raster_fixture()
with rasterio.open(path) as src:
    patch=src.read(window=Window(0,0,128,128))
    print(src.crs,src.transform,src.bounds)
    print('Upper-left pixel center:',src.xy(0,0))
assert patch.shape == (3,128,128)
batch=patch[np.newaxis].astype('float32')/10000
assert batch.shape == (1,3,128,128)
plt.imshow(batch[0].transpose(1,2,0));plt.title('Synthetic RGB noise — shape exercise only');plt.show()


## Your turn

Extract a 64×64 window starting at column 32, row 16. Find its upper-left pixel center. Predict the shape after selecting two bands.

Use a new cell below to try this before opening the solution.


In [ ]:
# Write your experiment here. The worked example above is already complete.


<details><summary>Solution and reasoning — open after trying</summary>

Use `Window(32,16,64,64)` and `src.xy(16,32)`. The tensor is (1,3,64,64), then (1,2,64,64). A model configured for three channels cannot accept two without modifying its input layer.

</details>


## Check yourself

What metadata disappears when you convert a raster to a NumPy array? Expected: EPSG:32613, 10 m pixels, and a (1,3,128,128) batch. Explain why a feature vector is not a burn probability.

**Exit ticket:** write one result, one explanation, and one limitation. Save the notebook with your own observations.


## Transfer to real data

For real Sentinel-2 input, preserve RGB order B04/B03/B02, masks, reflectance scaling, acquisition date and model preprocessing. Inspect the weights metadata before inference. A synthetic forward pass tests plumbing only.


## Optional TorchGeo lab

Set `RUN_TORCHGEO=True` after installing model dependencies. Set `PRETRAINED=True` to download weights. Random weights are explicitly labeled and are not a substitute for pretrained inference. This example targets TorchGeo 0.9.


In [ ]:
RUN_TORCHGEO = False
PRETRAINED = False
if RUN_TORCHGEO:
    import torch
    from torchgeo.datasets import RasterDataset
    from torchgeo.samplers import GridGeoSampler, Units
    from torchgeo.models import resnet50, ResNet50_Weights
    class TeachingRaster(RasterDataset):
        filename_glob='synthetic_rgb.tif'
        all_bands=('red','green','blue')
        rgb_bands=all_bands
    dataset=TeachingRaster(paths=str(OUT))
    sampler=GridGeoSampler(dataset,size=128,stride=128,units=Units.PIXELS)
    sample=dataset[next(iter(sampler))]
    image=sample['image'].unsqueeze(0).float()
    weights=ResNet50_Weights.SENTINEL2_RGB_MOCO if PRETRAINED else None
    torch.manual_seed(42)
    model=resnet50(weights=weights,in_chans=3)
    model.fc=torch.nn.Identity()
    model.eval()
    image=weights.transforms()(image) if weights is not None else image/10000
    with torch.inference_mode():
        features=model(image)
    print('PRETRAINED' if PRETRAINED else 'RANDOM WEIGHTS: mechanics only',features.shape)
    assert features.shape[0] == 1
    try:
        with torch.inference_mode(): model(image[:,:2])
    except RuntimeError as exc:
        print('Expected channel mismatch:',str(exc)[:180])
else:
    print('Optional TorchGeo/model lab not run; raster mechanics above are complete.')
